In [1]:
import json
import re
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
import pandas as pd

# Load JSON log entries from file
with open("datas/tanner_report.json") as f:
    logs = [json.loads(line) for line in f.readlines()]

In [9]:
# Define attack patterns using regex
patterns = {
    "RFI": r".*\b(http|https|ftp|ftps):\/\/.*",
    "LFI": r".*(\.\./)*(home|proc|usr|etc)/.*",
    "XSS": r".*<[^>]+>.*",
    "SQLi": r".*\b(union\s+select|select\s+.*\s+from|insert\s+into|drop\s+table|--|\bor\b\s+\d+=\d+).*",
    "Command Injection": r".*\b(alias|cat|cd|cp|echo|exec|find|for|grep|ifconfig|ls|man|mkdir|netstat|ping|ps|pwd|uname|wget|touch|while)\b.*",
    "PHP Code Injection": r".*\b(eval\(|assert\(|base64_decode\().*",
    "PHP Object Injection": r".*[\{;]?\s*O:\d+:\".*?\":\d+:\{.*?\}.*",
    "CRLF": r".*(%0d%0a|\r\n).*",
    "XXE": r".*<\?(xml|!DOCTYPE).*?>.*",
    "Template Injection": r".*(\{\{.*?\}\}|\{%.*?%\}).*"
}

In [10]:
# Function to match patterns in path and query
def detect_attacks(entry):
    text = entry.get("path", "")
    matches = []
    for attack, regex in patterns.items():
        if re.search(regex, text, re.IGNORECASE):
            matches.append(attack)
    return matches

# Extract and label data
data = defaultdict(lambda: {"entries": [], "attacks": set()})
for log in logs:
    sess_uuid = log.get("cookies", {}).get("sess_uuid")
    if sess_uuid:
        attacks = detect_attacks(log)
        data[sess_uuid]["entries"].append(log["path"])
        data[sess_uuid]["attacks"].update(attacks)

# Prepare dataset for clustering
sess_uuids = []
texts = []
labels = []
for uuid, content in data.items():
    sess_uuids.append(uuid)
    texts.append(" ".join(content["entries"]))
    labels.append(", ".join(content["attacks"]) if content["attacks"] else "Normal")

In [11]:
# Vectorize text using TF-IDF
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(texts)

# Apply KMeans clustering
kmeans = KMeans(n_clusters=3, random_state=42)
clusters = kmeans.fit_predict(X)

# Output results
results_df = pd.DataFrame({
    "sess_uuid": sess_uuids,
    "detected_attacks": labels,
    "cluster": clusters
})

In [12]:
results_df.to_csv("classified_sessions.csv", index=False)
print(results_df.head())

                              sess_uuid detected_attacks  cluster
0  54378fb4-cfdb-4038-95a7-884f04c507ca           Normal        2
1  31dc3b6c-7500-46ef-b693-2dd98beb42ad           Normal        0
2  96060529-1172-409f-9375-7d5184893fa2              LFI        1
3  4db3492d-8906-4698-80f8-f45fd620d10e           Normal        0
